# Действительно ли временная валидация важна?

Вопрос простой и честный: мы решили доверять временному протоколу, потому что тест сдвинут в
будущее. Но **разница между протоколами могла возникнуть совсем по другой причине** — во
временных фолдах модель обучается на меньшем числе строк. Если весь разрыв объясняется объёмом
данных, то никакой особой «временной сложности» нет, и мы зря перестраивали выводы.

Этот ноутбук ставит эксперимент, который разводит две причины.

## Как устроен контроль

Обычное сравнение «случайный протокол против временного» нечестное сразу по двум осям:
разные обучающие выборки **и** разные проверочные. Поэтому здесь три протокола, у которых
**валидационные строки одни и те же** — меняется только то, откуда берётся обучение.

| протокол | валидация | обучение |
|---|---|---|
| **A. хронология** | дни 14–15, 16–17, 18–19 | только **предыдущие** дни, N строк |
| **B. случайный, тот же объём** | те же самые строки | **N случайных** строк откуда угодно |
| **C. случайный, весь остаток** | те же самые строки | **все** оставшиеся строки |

Тогда две разницы отвечают каждая на свой вопрос:

- **B − A** — цена хронологии. Объём обучения одинаковый, отличается только то, что в A данные
  строго из прошлого, а в B — откуда угодно, включая будущее. Это и есть чистый эффект времени.
- **C − B** — цена объёма. Выбор строк в обоих случаях случайный, отличается только количество.

Если **B − A ≈ 0**, то временная валидация ничем не особенна, и можно спокойно пользоваться
случайной — она точнее, потому что оценивает больше строк.
Если **B − A заметно больше нуля**, перенос во времени действительно тяжелее, и временной
протокол обязателен.

## Что ещё проверяем заодно

Четыре набора признаков × две модели, чтобы вывод не зависел от одной конкретной конфигурации:

- **сырые** (6) — объёмные счётчики, примерно уровень quickstart;
- **универсальные** (33) — база из `solution.ipynb`;
- **отобранные под случайный** — жадный отбор групп с критерием по случайному протоколу;
- **отобранные под временной** — тот же отбор, но критерий по временному.

Последние два дают ответ на отдельный вопрос: **портит ли отбор признаков по случайному
протоколу качество на временном?**

Модели — XGBoost и CatBoost, два лидера предыдущего этапа.

Полный прогон — примерно 10 минут.

In [ ]:
# Настройки, данные, признаки. Общий код берём из avito_lib, чтобы не плодить копии.

import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

import avito_lib as L
from metric import precision_at_recall

SEED = 42
np.random.seed(SEED)
warnings.filterwarnings("ignore", message=".*Falling back to prediction using DMatrix.*")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

train, test, events = L.load_data()
ytr = train["target"].values.astype(int)

t0 = time.time()
pop = L.build_population_stats(events, train, test)
Xtr, GROUPS = L.build_features(train, events, pop)
print(f"признаки посчитаны за {time.time() - t0:.0f} c: {Xtr.shape}")

RAW = L.RAW_FEATURES
BASE = GROUPS["base"]
EXTRA = {k: v for k, v in GROUPS.items() if k != "base"}
print("сырые:", len(RAW), "| база:", len(BASE), "| групп-кандидатов:", len(EXTRA))

## Протоколы

Ниже строятся три набора фолдов. Главное в коде — **валидационные индексы во всех трёх
одинаковые**. Это то, что делает сравнение честным: никакой разницы в том, кого мы оцениваем,
только в том, на чём учились.

In [ ]:
# Три протокола с общими валидационными строками + обычный случайный для преемственности.

TIME_SPEC = [
    ("2026-04-06", "2026-04-13", "2026-04-14", "2026-04-15"),
    ("2026-04-06", "2026-04-15", "2026-04-16", "2026-04-17"),
    ("2026-04-06", "2026-04-17", "2026-04-18", "2026-04-19"),
]

day = train["window_start_ts"].dt.normalize()

# A. хронология: учимся строго на прошлом
folds_time = []
for a, b, c, d in TIME_SPEC:
    tr = np.where(((day >= a) & (day <= b)).values)[0]
    va = np.where(((day >= c) & (day <= d)).values)[0]
    folds_time.append((tr, va))

rng = np.random.default_rng(SEED)
all_idx = np.arange(len(ytr))

# B. случайный того же объёма: валидация та же, обучение — столько же строк, но откуда угодно
folds_matched = []
for tr, va in folds_time:
    rest = np.setdiff1d(all_idx, va)
    folds_matched.append((rng.choice(rest, len(tr), replace=False), va))

# C. случайный со всем остатком: валидация та же, обучение — всё, что не в валидации
folds_full = [(np.setdiff1d(all_idx, va), va) for _, va in folds_time]

# D. обычная 5-фолдовая кросс-валидация — для связи с прошлыми результатами
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
folds_cv5 = list(skf.split(Xtr, ytr))

PROTOCOLS = {
    "A. хронология": folds_time,
    "B. случайный, тот же объём": folds_matched,
    "C. случайный, весь остаток": folds_full,
    "D. обычный 5-fold": folds_cv5,
}

rows = []
for name, folds in PROTOCOLS.items():
    for i, (tr, va) in enumerate(folds, 1):
        rows.append({"протокол": name, "фолд": i, "обучение": len(tr), "валидация": len(va),
                     "ботов в валидации": int(ytr[va].sum()),
                     "доля ботов": round(float(ytr[va].mean()), 4)})
info = pd.DataFrame(rows)
print("Состав протоколов:")
display(info)

# проверяем главное свойство: у A, B и C валидация совпадает строка в строку
for i in range(3):
    assert np.array_equal(folds_time[i][1], folds_matched[i][1])
    assert np.array_equal(folds_time[i][1], folds_full[i][1])
print()
print("проверка пройдена: у протоколов A, B и C валидационные строки идентичны,")
print("различается только обучающая выборка — значит разницу в метрике создаёт именно она")

## Наборы признаков

Два первых набора фиксированы, два последних — результат жадного отбора групп, где критерий
приёма считается **по разным протоколам**. Отбор идёт одной и той же процедурой, отличается
только то, по какой метрике принимается решение.

Это отдельная проверка: если отбор под случайный протокол даёт на временном заметно худший
набор, значит перекос критерия стоит реальных денег, а не только красивой таблицы.

In [ ]:
# Отбор групп признаков двумя критериями.

def make_xgb(seed=SEED):
    return XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                         subsample=0.8, colsample_bytree=0.8, device="cpu",
                         tree_method="hist", eval_metric="logloss",
                         random_state=seed, verbosity=0, n_jobs=-1)


def make_cat(seed=SEED):
    return CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                              task_type="CPU", random_seed=seed, verbose=0,
                              allow_writing_files=False)


def oof_score(make_model, cols, folds):
    """Обучает модель на каждом фолде и считает метрику по собранным предсказаниям."""
    oof = np.full(len(ytr), np.nan)
    for tr, va in folds:
        m = make_model()
        m.fit(Xtr.iloc[tr][cols], ytr[tr])
        oof[va] = m.predict_proba(Xtr.iloc[va][cols])[:, 1]
    seen = ~np.isnan(oof)
    return precision_at_recall(ytr[seen], oof[seen]), oof


GROUP_ORDER = ["content", "rhythm", "navquery", "pointer", "sequence", "uaplat"]


def greedy_select(folds, label):
    """Жадно добавляет группы, пока метрика на ЭТОМ протоколе растёт."""
    cols = list(BASE)
    best, _ = oof_score(make_xgb, cols, folds)
    log = [{"шаг": "база", "признаков": len(cols), "метрика": round(best, 4), "решение": "старт"}]
    for gname in tqdm(GROUP_ORDER, desc=label, leave=False):
        cand = cols + [c for c in EXTRA[gname] if c not in cols]
        sc, _ = oof_score(make_xgb, cand, folds)
        take = sc > best
        log.append({"шаг": "+ " + gname, "признаков": len(cand), "метрика": round(sc, 4),
                    "решение": "принято" if take else "отклонено"})
        if take:
            cols, best = cand, sc
    return cols, pd.DataFrame(log)


t0 = time.time()
SEL_RANDOM, log_rnd = greedy_select(folds_cv5, "отбор под случайный")
SEL_TIME, log_time = greedy_select(folds_time, "отбор под временной")
print(f"отбор занял {time.time() - t0:.0f} c")
print()
print("Отбор с критерием по СЛУЧАЙНОМУ протоколу:")
display(log_rnd)
print("Отбор с критерием по ВРЕМЕННОМУ протоколу:")
display(log_time)

FEATURE_SETS = {
    "сырые (6)": RAW,
    "универсальные (33)": BASE,
    f"под случайный ({len(SEL_RANDOM)})": SEL_RANDOM,
    f"под временной ({len(SEL_TIME)})": SEL_TIME,
}

only_time = sorted(set(SEL_TIME) - set(SEL_RANDOM))
only_rnd = sorted(set(SEL_RANDOM) - set(SEL_TIME))
print()
print(f"наборы различаются: только во временном {len(only_time)} признаков, "
      f"только в случайном {len(only_rnd)}")
if only_time:
    print("  есть только в наборе под временной:", ", ".join(only_time[:12]))
if only_rnd:
    print("  есть только в наборе под случайный:", ", ".join(only_rnd[:12]))

## Полный прогон

4 набора признаков × 2 модели × 4 протокола. Каждая клетка — независимое обучение,
результат — метрика на собранных out-of-fold предсказаниях.

In [ ]:
# Факторный эксперимент. Примерно 8 минут, дольше всего идёт CatBoost.

MODELS = {"XGBoost": make_xgb, "CatBoost": make_cat}

results, oofs = [], {}
combos = [(fs, mn, pn) for fs in FEATURE_SETS for mn in MODELS for pn in PROTOCOLS]

t0 = time.time()
for fs_name, m_name, p_name in tqdm(combos, desc="эксперимент"):
    cols = FEATURE_SETS[fs_name]
    sc, oof = oof_score(MODELS[m_name], cols, PROTOCOLS[p_name])
    results.append({"признаки": fs_name, "модель": m_name, "протокол": p_name,
                    "P@R": round(sc, 4)})
    oofs[(fs_name, m_name, p_name)] = oof

R = pd.DataFrame(results)
print(f"готово за {time.time() - t0:.0f} c")

pivot = R.pivot_table(index=["признаки", "модель"], columns="протокол", values="P@R")
pivot = pivot[list(PROTOCOLS.keys())]
print()
print("МЕТРИКА P@R>=0.70 ПО ВСЕМ КЛЕТКАМ")
display(pivot)

## Разложение разрыва: хронология или объём?

Теперь главное. Для каждой клетки считаем две разницы, и обе — **на одних и тех же строках**,
что делает их гораздо точнее, чем сами метрики.

- **цена хронологии = B − A.** Объём обучения одинаковый. Отличается только то, что в A данные
  строго из прошлого, а в B взяты откуда угодно. Это чистый эффект времени.
- **цена объёма = C − B.** Строки в обоих случаях случайные, отличается только их количество.

К каждой разнице считаем bootstrap: 400 пересборок валидации, на каждой обе метрики сразу.
Доля пересборок, где разница положительна, — это и есть надёжность вывода.

In [ ]:
# Разложение с парным bootstrap.

def paired_delta(oof_a, oof_b, n_boot=400, seed=SEED):
    """Распределение разницы метрик двух моделей на одних и тех же пересобранных строках."""
    mask = ~(np.isnan(oof_a) | np.isnan(oof_b))
    y_, a_, b_ = ytr[mask], oof_a[mask], oof_b[mask]
    rng = np.random.default_rng(seed)
    n = len(y_)
    d = np.empty(n_boot)
    for i in range(n_boot):
        ix = rng.integers(0, n, n)
        ys = y_[ix]
        d[i] = precision_at_recall(ys, b_[ix]) - precision_at_recall(ys, a_[ix])
    return d


A, B, C = "A. хронология", "B. случайный, тот же объём", "C. случайный, весь остаток"

dec = []
for fs in FEATURE_SETS:
    for mn in MODELS:
        oA, oB, oC = oofs[(fs, mn, A)], oofs[(fs, mn, B)], oofs[(fs, mn, C)]
        d_chrono = paired_delta(oA, oB)
        d_size = paired_delta(oB, oC)
        dec.append({
            "признаки": fs, "модель": mn,
            "A хронология": round(precision_at_recall(ytr[~np.isnan(oA)], oA[~np.isnan(oA)]), 4),
            "B тот же объём": round(precision_at_recall(ytr[~np.isnan(oB)], oB[~np.isnan(oB)]), 4),
            "C весь остаток": round(precision_at_recall(ytr[~np.isnan(oC)], oC[~np.isnan(oC)]), 4),
            "цена хронологии": round(float(d_chrono.mean()), 4),
            "надёжность хрон.": round(float((d_chrono > 0).mean()), 2),
            "цена объёма": round(float(d_size.mean()), 4),
            "надёжность объёма": round(float((d_size > 0).mean()), 2),
        })

D = pd.DataFrame(dec)
print("РАЗЛОЖЕНИЕ РАЗРЫВА ПО ПРИЧИНАМ")
print("(надёжность — доля из 400 пересборок, где разница положительна; 0.90 и выше = уверенно)")
display(D)

print()
print("Усреднённо по всем восьми конфигурациям:")
summary = pd.DataFrame({
    "величина": ["цена хронологии (B-A)", "цена объёма (C-B)"],
    "среднее": [round(D["цена хронологии"].mean(), 4), round(D["цена объёма"].mean(), 4)],
    "минимум": [round(D["цена хронологии"].min(), 4), round(D["цена объёма"].min(), 4)],
    "максимум": [round(D["цена хронологии"].max(), 4), round(D["цена объёма"].max(), 4)],
    "уверенных клеток из 8": [int((D["надёжность хрон."] >= 0.9).sum()),
                              int((D["надёжность объёма"] >= 0.9).sum())],
})
display(summary)

In [ ]:
# Картинка: та же таблица, но глазами.

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
order = list(PROTOCOLS.keys())
colors = ["#c0392b", "#e59866", "#5dade2", "#95a5a6"]

for ax, mn in zip(axes, MODELS):
    sub = R[R["модель"] == mn].pivot_table(index="признаки", columns="протокол", values="P@R")
    sub = sub.loc[list(FEATURE_SETS.keys()), order]
    x = np.arange(len(sub))
    w = 0.2
    for j, p in enumerate(order):
        ax.bar(x + (j - 1.5) * w, sub[p].values, w, label=p, color=colors[j])
    ax.set_xticks(x)
    ax.set_xticklabels(sub.index, rotation=20, ha="right")
    ax.set_title(mn)
    ax.grid(axis="y", alpha=0.3)
    ax.set_ylabel("P@R >= 0.70")

axes[0].legend(fontsize=9, loc="upper left")
plt.suptitle("Одни и те же валидационные строки, разное обучение", y=1.02)
plt.tight_layout()
plt.show()

## Отдельный вопрос: портит ли отбор признаков по случайному протоколу?

Сравниваем два набора, отобранных одной процедурой с разными критериями, — и смотрим на них
там, где это важно, то есть на протоколе A (хронология).

In [ ]:
# Набор "под случайный" против набора "под временной", оба на хронологическом протоколе.

fs_rnd = [k for k in FEATURE_SETS if k.startswith("под случайный")][0]
fs_time = [k for k in FEATURE_SETS if k.startswith("под временной")][0]

comp = []
for mn in MODELS:
    d = paired_delta(oofs[(fs_rnd, mn, A)], oofs[(fs_time, mn, A)])
    comp.append({
        "модель": mn,
        "набор под случайный": R[(R["признаки"] == fs_rnd) & (R["модель"] == mn)
                                 & (R["протокол"] == A)]["P@R"].iat[0],
        "набор под временной": R[(R["признаки"] == fs_time) & (R["модель"] == mn)
                                 & (R["протокол"] == A)]["P@R"].iat[0],
        "разница": round(float(d.mean()), 4),
        "надёжность": round(float((d > 0).mean()), 2),
    })
CMP = pd.DataFrame(comp)
print("Оба набора оценены на хронологическом протоколе (то, что имитирует реальный тест):")
display(CMP)

## Вывод

In [ ]:
# Вердикт, собранный из посчитанных чисел. Ничего не придумано, всё считается выше.

chrono = D["цена хронологии"].mean()
size = D["цена объёма"].mean()
n_ch = int((D["надёжность хрон."] >= 0.9).sum())
n_sz = int((D["надёжность объёма"] >= 0.9).sum())
NOISE = 0.041

line = "=" * 78
print(line)
print("  ОТВЕТ НА ВОПРОС: ВАЖНА ЛИ ИМЕННО ВРЕМЕННАЯ ВАЛИДАЦИЯ")
print(line)
print()
print(f"  цена хронологии (B-A): {chrono:+.4f}   уверенных клеток {n_ch} из 8")
print(f"  цена объёма     (C-B): {size:+.4f}   уверенных клеток {n_sz} из 8")
print(f"  для сравнения, шум метрики: {NOISE:.4f}")
print()

if abs(chrono) < NOISE / 2 and n_ch <= 2:
    print("  ХРОНОЛОГИЯ ПОЧТИ НЕ ВЛИЯЕТ.")
    print("  При одинаковом объёме обучения неважно, берутся строки из прошлого или откуда")
    print("  угодно. Значит разрыв между протоколами, который мы видели раньше, объяснялся")
    print("  объёмом данных, а не сдвигом во времени.")
    print("  Практический вывод: можно опираться на случайный протокол — он точнее, потому")
    print("  что оценивает вдвое больше строк. Временной оставить как страховку.")
else:
    print("  ХРОНОЛОГИЯ ВЛИЯЕТ.")
    print("  Даже при одинаковом объёме обучения модель, учившаяся строго на прошлом,")
    print("  работает хуже. Значит перенос во времени действительно тяжелее, и случайный")
    print("  протокол систематически завышает оценку.")
    print("  Практический вывод: решения принимаем по временному протоколу.")
print()

if abs(size) > abs(chrono) * 2 and n_sz >= 4:
    print(f"  Заодно видно, что объём обучения важнее хронологии примерно "
          f"в {abs(size / chrono):.1f} раза.")
    print("  Значит финальную модель точно надо учить на ВСЁМ train, а не на его части.")
print()
print("  Про отбор признаков:")
for _, r in CMP.iterrows():
    verdict = "набор под временной лучше" if r["разница"] > 0 else "разницы нет или хуже"
    print(f"    {r['модель']:10s} разница {r['разница']:+.4f}, надёжность {r['надёжность']:.2f}"
          f"  -> {verdict}")
print(line)

## Пятая ветка: компоненты PCA поверх отобранных признаков

Зачем это вообще может помочь, хотя PCA и деревья обычно не дружат.

Дерево делит пространство **только по осям** — одно условие на один признак за раз. Линейную
комбинацию вроде «0.3 × энтропия категорий + 0.7 × глубина выдачи» бустинг способен изобразить
лишь лесенкой из десятков разбиений, и то приблизительно. Компоненты PCA дают такие комбинации
готовыми, то есть добавляют ровно то, что дерево само построить почти не может.

Обратная сторона: компонент — это плотная смесь всех признаков сразу, её невозможно объяснить
словами. Если прирост будет в пределах шума, брать не стоит: потеряем в понятности решения, а
понятность здесь отдельно оценивается.

### Как избежать утечки

PCA — это обучаемое преобразование: он смотрит на данные и находит направления максимального
разброса. Если обучить его на всём train, в каждый фолд просочится информация из его же
валидационной части.

Поэтому вся цепочка — заполнение пропусков медианой, стандартизация, PCA — **обучается внутри
каждого фолда только на обучающей половине** и лишь затем применяется к валидационной. Тот же
принцип, что и с популярностью товара: преобразование не имеет права видеть то, что мы
собираемся предсказывать.

Проверяем три размерности: 5, 10 и 20 компонент.

In [ ]:
# PCA, обучаемый строго внутри фолда.

from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def oof_score_pca(make_model, cols, folds, n_comp, seed=SEED):
    """То же, что oof_score, но к признакам добавляются компоненты PCA.

    Цепочка impute -> scale -> PCA обучается ТОЛЬКО на обучающей части фолда.
    """
    oof = np.full(len(ytr), np.nan)
    evr = []
    pc_names = [f"pc{i + 1}" for i in range(n_comp)]
    for tr, va in folds:
        Xa, Xb = Xtr.iloc[tr][cols], Xtr.iloc[va][cols]
        pipe = make_pipeline(SimpleImputer(strategy="median"),
                             StandardScaler(),
                             PCA(n_components=n_comp, random_state=seed))
        Pa = pipe.fit_transform(Xa)
        Pb = pipe.transform(Xb)
        evr.append(pipe[-1].explained_variance_ratio_.sum())
        Xa2 = pd.concat([Xa.reset_index(drop=True), pd.DataFrame(Pa, columns=pc_names)], axis=1)
        Xb2 = pd.concat([Xb.reset_index(drop=True), pd.DataFrame(Pb, columns=pc_names)], axis=1)
        m = make_model()
        m.fit(Xa2, ytr[tr])
        oof[va] = m.predict_proba(Xb2)[:, 1]
    seen = ~np.isnan(oof)
    return precision_at_recall(ytr[seen], oof[seen]), oof, float(np.mean(evr))


BEST_COLS = SEL_RANDOM
PCA_PROTOCOLS = {"A. хронология": folds_time, "D. обычный 5-fold": folds_cv5}
N_COMPS = [5, 10, 20]

pca_rows, pca_oofs = [], {}
t0 = time.time()
combos = [(mn, pn, k) for mn in MODELS for pn in PCA_PROTOCOLS for k in N_COMPS]
for m_name, p_name, k in tqdm(combos, desc="PCA-ветка"):
    sc, oof, evr = oof_score_pca(MODELS[m_name], BEST_COLS, PCA_PROTOCOLS[p_name], k)
    pca_rows.append({"модель": m_name, "протокол": p_name, "компонент": k,
                     "P@R": round(sc, 4), "объяснённая дисперсия": round(evr, 3)})
    pca_oofs[(m_name, p_name, k)] = oof
print(f"готово за {time.time() - t0:.0f} c")

P = pd.DataFrame(pca_rows)

# базовые значения без PCA — из основного эксперимента
base_rows = []
for m_name in MODELS:
    for p_name in PCA_PROTOCOLS:
        base_rows.append({"модель": m_name, "протокол": p_name, "компонент": 0,
                          "P@R": R[(R["признаки"] == fs_rnd) & (R["модель"] == m_name)
                                   & (R["протокол"] == p_name)]["P@R"].iat[0],
                          "объяснённая дисперсия": np.nan})
P = pd.concat([pd.DataFrame(base_rows), P], ignore_index=True)

print()
print("PCA ПОВЕРХ 83 ПРИЗНАКОВ (компонент = 0 означает без PCA)")
display(P.pivot_table(index=["модель", "протокол"], columns="компонент", values="P@R"))
print()
print("Сколько дисперсии исходных признаков ловят компоненты:")
display(P.dropna(subset=["объяснённая дисперсия"])
         .groupby("компонент")["объяснённая дисперсия"].mean().round(3).to_frame("доля"))

In [ ]:
# Вердикт по PCA: парное сравнение с вариантом без компонент.

pca_verdict = []
for m_name in MODELS:
    for p_name in PCA_PROTOCOLS:
        base_oof = oofs[(fs_rnd, m_name, p_name)]
        for k in N_COMPS:
            d = paired_delta(base_oof, pca_oofs[(m_name, p_name, k)])
            pca_verdict.append({
                "модель": m_name, "протокол": p_name, "компонент": k,
                "разница": round(float(d.mean()), 4),
                "надёжность": round(float((d > 0).mean()), 2),
                "интервал 5-95%": f"[{np.percentile(d, 5):+.4f}, {np.percentile(d, 95):+.4f}]",
            })
V = pd.DataFrame(pca_verdict)
print("Парное сравнение: с компонентами против без них, на одних и тех же строках")
display(V)

best = V.loc[V["разница"].idxmax()]
n_sure = int((V["надёжность"] >= 0.9).sum())
line = "=" * 78
print()
print(line)
print("  ВЕРДИКТ ПО PCA")
print(line)
print(f"  средняя разница по всем 12 клеткам: {V['разница'].mean():+.4f}")
print(f"  клеток с надёжностью 0.90 и выше:   {n_sure} из {len(V)}")
print(f"  лучшая клетка: {best['модель']}, {best['протокол']}, "
      f"{best['компонент']} компонент -> {best['разница']:+.4f} "
      f"при надёжности {best['надёжность']:.2f}")
print()
if n_sure >= len(V) // 2 and V["разница"].mean() > 0:
    print("  КОМПОНЕНТЫ ПОМОГАЮТ. Прирост держится в большинстве клеток — значит это не")
    print("  случайность одной конфигурации. Имеет смысл добавить их в финальный набор.")
elif V["разница"].mean() > 0 and n_sure > 0:
    print("  РЕЗУЛЬТАТ НЕОДНОЗНАЧНЫЙ. Прирост есть, но держится не везде. За него придётся")
    print("  заплатить понятностью решения: компонента — плотная смесь всех признаков,")
    print("  объяснить её проверяющему словами не получится. Решай по приоритетам.")
else:
    print("  КОМПОНЕНТЫ НЕ ПОМОГАЮТ. Бустинг и без них вытаскивает всё, что есть в данных.")
    print("  Не берём: понятность решения дороже прироста, которого нет.")
print(line)